In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import joblib
import warnings
from tqdm import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')

# 1. Data Loading and Preprocessing
def load_and_preprocess():
    print("Loading and preprocessing data...")
    data = pd.read_csv('full_data_unhealthy_imputed.csv')
    
    # Validate required columns
    required_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour', 'date', 
                    'oestrus', 'calving', 'lameness', 'mastitis', 'cow']
    missing_cols = set(required_cols) - set(data.columns)
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")
    
    # Feature engineering
    rest_arr = data['REST'].values
    eat_arr = data['EAT'].values
    
    # Calculate ratio safely
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(
            (eat_arr == 0) & (rest_arr == 0),
            1.0,
            rest_arr / np.where(eat_arr == 0, np.nan, eat_arr)
        )
    data['rest_to_eat_ratio'] = pd.Series(ratio).fillna(1.0)
    
    # Cyclical time features
    hour_arr = data['hour'].values
    data['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
    data['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
    
    features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
               'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
    targets = ['oestrus', 'calving', 'lameness', 'mastitis']
    
    # Clean data
    data = data.dropna(subset=features + targets + ['cow'])
    
    # Sort by cow, date, and hour to create sequences
    data = data.sort_values(['cow', 'date', 'hour'])
    
    return data[features], data[targets], data['cow']

# 2. Create sequences for LSTM
def create_sequences(X, y, groups, sequence_length=24):
    """
    Create sequences of fixed length for LSTM training
    Returns:
        X_seq: 3D array of shape (n_sequences, sequence_length, n_features)
        y_seq: 2D array of shape (n_sequences, n_targets)
        groups_seq: 1D array of group IDs for each sequence
    """
    X_seq = []
    y_seq = []
    groups_seq = []
    
    for cow_id in tqdm(groups.unique(), desc="Creating sequences"):
        cow_mask = (groups == cow_id)
        X_cow = X[cow_mask].values
        y_cow = y[cow_mask].values
        
        # Create sequences for each cow
        for i in range(len(X_cow) - sequence_length):
            X_seq.append(X_cow[i:i+sequence_length])
            y_seq.append(y_cow[i+sequence_length-1])  # Use last element in sequence as label
            groups_seq.append(cow_id)
            
    return np.array(X_seq), np.array(y_seq), np.array(groups_seq)

# 3. Calculate class weights
def calculate_class_weights(y):
    class_weights = []
    for i in range(y.shape[1]):
        class_counts = np.bincount(y[:, i].astype(int))
        if len(class_counts) < 2:
            class_counts = np.append(class_counts, [0])
        weight_for_0 = (1 / class_counts[0]) * (len(y) / 2.0)
        weight_for_1 = (1 / class_counts[1]) * (len(y) / 2.0)
        class_weights.append({0: weight_for_0, 1: weight_for_1})
    return class_weights

# 4. Build LSTM model
def build_lstm_model(input_shape, num_targets):
    model = Sequential([
        Bidirectional(LSTM(128, return_sequences=True, input_shape=input_shape)),
        Dropout(0.3),
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(num_targets, activation='sigmoid')
    ])
    
    model.compile(optimizer=Adam(learning_rate=0.001),
                 loss='binary_crossentropy',
                 metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    
    return model

# 5. Train and evaluate with GroupKFold
def train_and_evaluate():
    # Load data
    X, y, groups = load_and_preprocess()
    print(f"Data shape: {X.shape}, Targets shape: {y.shape}")
    
    # Create sequences
    SEQUENCE_LENGTH = 24  # 24-hour sequences
    X_seq, y_seq, groups_seq = create_sequences(X, y, groups, SEQUENCE_LENGTH)
    print(f"Sequenced data shape: {X_seq.shape}, Target shape: {y_seq.shape}")
    
    # Scale features
    scaler = StandardScaler()
    n_samples, n_timesteps, n_features = X_seq.shape
    X_scaled = scaler.fit_transform(X_seq.reshape(-1, n_features))
    X_scaled = X_scaled.reshape(n_samples, n_timesteps, n_features)
    
    # Calculate class weights
    class_weights = calculate_class_weights(y_seq)
    
    # GroupKFold cross-validation
    gkf = GroupKFold(n_splits=3)
    best_models = []
    best_thresholds = []
    all_metrics = []
    
    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_scaled, y_seq, groups=groups_seq)):
        print(f"\nTraining Fold {fold+1}/3")
        
        # Split data
        X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
        y_train, y_val = y_seq[train_idx], y_seq[val_idx]
        
        # Build model
        model = build_lstm_model((SEQUENCE_LENGTH, X_train.shape[2]), y_train.shape[1])
        
        # Callbacks
        callbacks = [
            EarlyStopping(monitor='val_auc', patience=5, mode='max', restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)
        ]
        
        # Train model
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=50,
            batch_size=256,
            callbacks=callbacks,
            verbose=1
        )
        
        # Predict probabilities
        y_proba = model.predict(X_val, verbose=0)
        
        # Find optimal thresholds
        thresholds = []
        for i in range(y_val.shape[1]):
            precision, recall, thresh = precision_recall_curve(y_val[:, i], y_proba[:, i])
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
            best_idx = np.argmax(f1_scores)
            thresholds.append(thresh[best_idx])
        
        best_thresholds.append(thresholds)
        
        # Evaluate
        y_pred = (y_proba > np.array(thresholds)).astype(int)
        
        print(f"\nFold {fold+1} Validation Performance:")
        metrics = []
        for i, target in enumerate(['oestrus', 'calving', 'lameness', 'mastitis']):
            report = classification_report(y_val[:, i], y_pred[:, i], output_dict=True, zero_division=0)
            auc = roc_auc_score(y_val[:, i], y_proba[:, i])
            print(f"{target} (AUC={auc:.3f}, Threshold={thresholds[i]:.3f}):")
            print(f"  Precision: {report['1']['precision']:.3f}, Recall: {report['1']['recall']:.3f}, F1: {report['1']['f1-score']:.3f}")
            metrics.append({
                'target': target,
                'auc': auc,
                'precision': report['1']['precision'],
                'recall': report['1']['recall'],
                'f1': report['1']['f1-score'],
                'threshold': thresholds[i]
            })
        
        all_metrics.append(metrics)
        best_models.append(model)
    
    # Select best model based on average F1 score
    avg_f1_scores = [
        np.mean([m['f1'] for m in metrics]) for metrics in all_metrics
    ]
    best_fold = np.argmax(avg_f1_scores)
    print(f"\nSelected best model from Fold {best_fold+1}")
    
    return best_models[best_fold], best_thresholds[best_fold], all_metrics[best_fold], scaler, SEQUENCE_LENGTH

# 6. Prediction Function
def predict(new_data, model, thresholds, scaler, sequence_length, features):
    """
    Predict using the LSTM model. Note: new_data should contain at least `sequence_length` hours
    of data for the same cow to form a complete sequence.
    """
    # Feature engineering
    X_new = new_data.copy()
    if {'EAT', 'REST'}.issubset(X_new.columns):
        rest_arr = X_new['REST'].values
        eat_arr = X_new['EAT'].values
        with np.errstate(divide='ignore', invalid='ignore'):
            ratio = np.where(
                (eat_arr == 0) & (rest_arr == 0),
                1.0,
                rest_arr / np.where(eat_arr == 0, np.nan, eat_arr)
            )
        X_new['rest_to_eat_ratio'] = pd.Series(ratio).fillna(1.0)
    
    if 'hour' in X_new.columns:
        hour_arr = X_new['hour'].values
        X_new['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
        X_new['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
    
    # Select features and scale
    X_new = X_new[features].values[-sequence_length:]  # Take last sequence_length hours
    if len(X_new) < sequence_length:
        # Pad with zeros if not enough data
        padding = np.zeros((sequence_length - len(X_new), len(features)))
        X_new = np.vstack([padding, X_new])
    
    X_scaled = scaler.transform(X_new)
    X_seq = X_scaled.reshape(1, sequence_length, len(features))
    
    # Predict probabilities
    y_proba = model.predict(X_seq, verbose=0)[0]
    
    # Apply thresholds
    results = {
        'oestrus': int(y_proba[0] > thresholds[0]),
        'oestrus_prob': y_proba[0],
        'calving': int(y_proba[1] > thresholds[1]),
        'calving_prob': y_proba[1],
        'lameness': int(y_proba[2] > thresholds[2]),
        'lameness_prob': y_proba[2],
        'mastitis': int(y_proba[3] > thresholds[3]),
        'mastitis_prob': y_proba[3]
    }
    
    return results

if __name__ == "__main__":
    try:
        print("Starting cattle health prediction with LSTM...")
        model, thresholds, metrics, scaler, seq_length = train_and_evaluate()
        
        # Save model
        model.save('lstm_cattle_model.keras')
        joblib.dump({
            'thresholds': thresholds,
            'scaler': scaler,
            'sequence_length': seq_length,
            'features': ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
                         'rest_to_eat_ratio', 'hour_sin', 'hour_cos'],
            'metrics': metrics
        }, 'lstm_cattle_model_params.pkl')
        print("\nLSTM model saved successfully!")
        
        # Print final metrics
        print("\nFinal Model Performance:")
        for metric in metrics:
            print(f"{metric['target']}: AUC={metric['auc']:.3f}, F1={metric['f1']:.3f}, "
                  f"Precision={metric['precision']:.3f}, Recall={metric['recall']:.3f}")
        
        # Example prediction (requires 24 hours of historical data)
        # In practice, you'd gather 24 hours of data for a cow
        test_data = pd.DataFrame({
            'IN_ALLEYS': np.random.uniform(0, 50, 24),
            'REST': np.random.uniform(2000, 4000, 24),
            'EAT': np.random.uniform(0, 1000, 24),
            'ACTIVITY_LEVEL': np.random.uniform(-1000, -500, 24),
            'hour': np.tile(np.arange(24), 1)[0]  # 24 consecutive hours
        })
        
        print("\nExample predictions for last hour in sequence:")
        print(predict(test_data, model, thresholds, scaler, seq_length, 
                      ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
                       'rest_to_eat_ratio', 'hour_sin', 'hour_cos']))
        
    except Exception as e:
        print(f"\nError: {str(e)}")

Starting cattle health prediction with LSTM...
Loading and preprocessing data...
Data shape: (1935667, 7), Targets shape: (1935667, 4)


Creating sequences: 100%|██████████| 279/279 [00:06<00:00, 40.43it/s]


Sequenced data shape: (1928971, 24, 7), Target shape: (1928971, 4)

Training Fold 1/3
Epoch 1/50
5024/5024 ━━━━━━━━━━━━━━━━━━━━ 2096s 413ms/step - accuracy: 0.5437 - auc: 0.7675 - loss: 0.0217 - val_accuracy: 0.4818 - val_auc: 0.8397 - val_loss: 0.0104 - learning_rate: 0.0010
Epoch 2/50
5024/5024 ━━━━━━━━━━━━━━━━━━━━ 2270s 442ms/step - accuracy: 0.5713 - auc: 0.8693 - loss: 0.0099 - val_accuracy: 0.6077 - val_auc: 0.8181 - val_loss: 0.0104 - learning_rate: 0.0010
Epoch 3/50
5024/5024 ━━━━━━━━━━━━━━━━━━━━ 1265s 248ms/step - accuracy: 0.4668 - auc: 0.9003 - loss: 0.0087 - val_accuracy: 0.2472 - val_auc: 0.7732 - val_loss: 0.0113 - learning_rate: 0.0010
Epoch 4/50
4338/5024 ━━━━━━━━━━━━━━━━━━━━ 2:21 206ms/step - accuracy: 0.3874 - auc: 0.9397 - loss: 0.0068

KeyboardInterrupt: 

In [13]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import joblib
import warnings
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Suppress warnings
warnings.filterwarnings("ignore")
tf.get_logger().setLevel('ERROR')

# 1. Data Loading and Preprocessing - FIXED
def load_and_preprocess():
    logger.info("Loading and preprocessing data...")
    try:
        data = pd.read_csv('full_data_unhealthy_imputed.csv')
        
        # Validate required columns
        required_cols = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'hour', 'date', 
                        'oestrus', 'calving', 'lameness', 'mastitis', 'cow']
        missing_cols = set(required_cols) - set(data.columns)
        if missing_cols:
            raise ValueError(f"Missing required columns: {missing_cols}")
        
        # Feature engineering - SAFE VERSION
        rest_arr = data['REST'].values
        eat_arr = data['EAT'].values
        
        # Calculate ratio safely using vectorized operations
        ratio = np.zeros(len(data))
        mask_both_zero = (eat_arr == 0) & (rest_arr == 0)
        mask_eat_zero = (eat_arr == 0) & (rest_arr != 0)
        mask_ok = (~mask_both_zero) & (~mask_eat_zero)
        
        ratio[mask_both_zero] = 1.0
        ratio[mask_eat_zero] = 100.0  # Large ratio when resting but not eating
        ratio[mask_ok] = rest_arr[mask_ok] / eat_arr[mask_ok]
        
        data['rest_to_eat_ratio'] = ratio
        
        # Cyclical time features
        hour_arr = data['hour'].values
        data['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
        data['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
        
        features = ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
                   'rest_to_eat_ratio', 'hour_sin', 'hour_cos']
        targets = ['oestrus', 'calving', 'lameness', 'mastitis']
        
        # Clean data
        data = data.dropna(subset=features + targets + ['cow'])
        logger.info(f"Loaded data shape: {data[features].shape}")
        
        return data[features], data[targets]
    
    except Exception as e:
        logger.error(f"Error in load_and_preprocess: {str(e)}")
        raise

# 2. Create sequences for LSTM
def create_sequences(X, y, sequence_length=12):
    """Create sequences from data"""
    logger.info("Creating sequences...")
    try:
        X_seq = []
        y_seq = []
        
        for i in range(len(X) - sequence_length):
            X_seq.append(X[i:i+sequence_length])
            y_seq.append(y[i+sequence_length-1])
            
        return np.array(X_seq), np.array(y_seq)
    except Exception as e:
        logger.error(f"Error in create_sequences: {str(e)}")
        raise

# 3. Build LSTM model
def build_lstm_model(input_shape, num_targets):
    logger.info("Building LSTM model...")
    try:
        model = Sequential([
            LSTM(64, input_shape=input_shape, return_sequences=False),
            Dense(32, activation='relu'),
            Dense(num_targets, activation='sigmoid')
        ])
        
        model.compile(optimizer=Adam(learning_rate=0.001),
                     loss='binary_crossentropy',
                     metrics=[tf.keras.metrics.AUC(name='auc')])
        
        model.summary(print_fn=logger.info)
        return model
    except Exception as e:
        logger.error(f"Error building model: {str(e)}")
        raise

# 4. Train and evaluate
def train_and_evaluate():
    try:
        # Load data
        X, y = load_and_preprocess()
        logger.info(f"Data shape: {X.shape}, Targets shape: {y.shape}")
        
        # Create sequences
        SEQUENCE_LENGTH = 12
        X_seq, y_seq = create_sequences(X.values, y.values, SEQUENCE_LENGTH)
        logger.info(f"Sequenced data shape: {X_seq.shape}, Target shape: {y_seq.shape}")
        
        # Scale features
        scaler = StandardScaler()
        n_samples, n_timesteps, n_features = X_seq.shape
        X_scaled = scaler.fit_transform(X_seq.reshape(-1, n_features))
        X_scaled = X_scaled.reshape(n_samples, n_timesteps, n_features)
        
        # Train-test split
        X_train, X_val, y_train, y_val = train_test_split(
            X_scaled, y_seq, test_size=0.2, random_state=42
        )
        logger.info(f"Train shape: {X_train.shape}, Validation shape: {X_val.shape}")
        
        # Build model
        model = build_lstm_model((SEQUENCE_LENGTH, n_features), y_train.shape[1])
        
        # Train model
        logger.info("Training model...")
        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=15,
            batch_size=512,
            callbacks=[EarlyStopping(monitor='val_auc', patience=3, mode='max', restore_best_weights=True)],
            verbose=1
        )
        
        # Predict probabilities
        y_proba = model.predict(X_val, verbose=0)
        
        # Find optimal thresholds
        thresholds = []
        for i in range(y_val.shape[1]):
            precision, recall, thresh = precision_recall_curve(y_val[:, i], y_proba[:, i])
            f1_scores = 2 * (precision * recall) / (precision + recall + 1e-6)
            best_idx = np.argmax(f1_scores)
            thresholds.append(thresh[best_idx])
        
        # Evaluate
        y_pred = (y_proba > np.array(thresholds)).astype(int)
        
        logger.info("\nValidation Performance:")
        metrics = []
        for i, target in enumerate(['oestrus', 'calving', 'lameness', 'mastitis']):
            report = classification_report(y_val[:, i], y_pred[:, i], output_dict=True, zero_division=0)
            auc = roc_auc_score(y_val[:, i], y_proba[:, i])
            logger.info(f"{target} (AUC={auc:.3f}, Threshold={thresholds[i]:.3f}):")
            logger.info(f"  Precision: {report['1']['precision']:.3f}, Recall: {report['1']['recall']:.3f}, F1: {report['1']['f1-score']:.3f}")
            metrics.append({
                'target': target,
                'auc': auc,
                'precision': report['1']['precision'],
                'recall': report['1']['recall'],
                'f1': report['1']['f1-score'],
                'threshold': thresholds[i]
            })
        
        # Save model
        model.save('lstm_cattle_model.keras')
        joblib.dump({
            'thresholds': thresholds,
            'scaler': scaler,
            'sequence_length': SEQUENCE_LENGTH,
            'features': X.columns.tolist(),
            'metrics': metrics
        }, 'lstm_cattle_model_params.pkl')
        logger.info("Model saved successfully!")
        
        return model, thresholds, metrics, scaler, SEQUENCE_LENGTH
    
    except Exception as e:
        logger.error(f"Error in train_and_evaluate: {str(e)}")
        raise

# 5. Prediction Function - FIXED
def predict(new_data, model, thresholds, scaler, sequence_length, features):
    try:
        # Convert to DataFrame if needed
        if not isinstance(new_data, pd.DataFrame):
            new_data = pd.DataFrame(new_data, columns=features)
            
        # Feature engineering - SAFE VERSION
        X_new = new_data.copy()
        
        # Handle rest_to_eat_ratio if REST and EAT are present
        if 'REST' in X_new.columns and 'EAT' in X_new.columns:
            rest_arr = X_new['REST'].values
            eat_arr = X_new['EAT'].values
            
            # Calculate ratio safely using vectorized operations
            ratio = np.zeros(len(X_new))
            mask_both_zero = (eat_arr == 0) & (rest_arr == 0)
            mask_eat_zero = (eat_arr == 0) & (rest_arr != 0)
            mask_ok = (~mask_both_zero) & (~mask_eat_zero)
            
            ratio[mask_both_zero] = 1.0
            ratio[mask_eat_zero] = 100.0
            ratio[mask_ok] = rest_arr[mask_ok] / eat_arr[mask_ok]
            
            X_new['rest_to_eat_ratio'] = ratio
        
        # Handle time features
        if 'hour' in X_new.columns:
            hour_arr = X_new['hour'].values
            X_new['hour_sin'] = np.sin(2 * np.pi * hour_arr / 24)
            X_new['hour_cos'] = np.cos(2 * np.pi * hour_arr / 24)
        
        # Ensure all features exist
        for feature in features:
            if feature not in X_new.columns:
                X_new[feature] = 0.0  # Default value
                logger.warning(f"Missing feature {feature} - using default value 0.0")
        
        # Select features and scale
        X_values = X_new[features].values[-sequence_length:]
        if len(X_values) < sequence_length:
            padding = np.zeros((sequence_length - len(X_values), len(features)))
            X_values = np.vstack([padding, X_values])
        
        X_scaled = scaler.transform(X_values)
        X_seq = X_scaled.reshape(1, sequence_length, len(features))
        
        # Predict probabilities
        y_proba = model.predict(X_seq, verbose=0)[0]
        
        # Apply thresholds
        results = {
            'oestrus': int(y_proba[0] > thresholds[0]),
            'oestrus_prob': float(y_proba[0]),
            'calving': int(y_proba[1] > thresholds[1]),
            'calving_prob': float(y_proba[1]),
            'lameness': int(y_proba[2] > thresholds[2]),
            'lameness_prob': float(y_proba[2]),
            'mastitis': int(y_proba[3] > thresholds[3]),
            'mastitis_prob': float(y_proba[3])
        }
        
        return results
    
    except Exception as e:
        logger.error(f"Error in predict: {str(e)}")
        raise

if __name__ == "__main__":
    try:
        logger.info("Starting cattle health prediction with LSTM...")
        model, thresholds, metrics, scaler, seq_length = train_and_evaluate()
        
        # Print final metrics
        if metrics:
            logger.info("\nFinal Model Performance:")
            for metric in metrics:
                logger.info(f"{metric['target']}: AUC={metric['auc']:.3f}, F1={metric['f1']:.3f}, "
                            f"Precision={metric['precision']:.3f}, Recall={metric['recall']:.3f}")
        
        # Example prediction - with proper feature setup
        test_data = pd.DataFrame({
            'IN_ALLEYS': np.random.uniform(0, 50, 12),
            'REST': np.random.uniform(2000, 4000, 12),
            'EAT': np.random.uniform(0, 1000, 12),
            'ACTIVITY_LEVEL': np.random.uniform(-1000, -500, 12),
            'hour': np.arange(12)
        })
        
        # Add required features that will be created in prediction
        test_data['rest_to_eat_ratio'] = 1.0  # Will be recalculated
        test_data['hour_sin'] = 0.0
        test_data['hour_cos'] = 0.0
        
        logger.info("\nExample predictions for last hour in sequence:")
        result = predict(test_data, model, thresholds, scaler, seq_length, 
                         ['IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 
                          'rest_to_eat_ratio', 'hour_sin', 'hour_cos'])
        logger.info(result)
        
    except Exception as e:
        logger.error(f"Fatal error: {str(e)}")

2025-07-02 15:14:55,681 - INFO - Starting cattle health prediction with LSTM...
2025-07-02 15:14:55,682 - INFO - Loading and preprocessing data...
2025-07-02 15:14:57,992 - INFO - Loaded data shape: (1935667, 7)
2025-07-02 15:14:58,056 - INFO - Data shape: (1935667, 7), Targets shape: (1935667, 4)
2025-07-02 15:14:58,056 - INFO - Creating sequences...
2025-07-02 15:15:00,742 - INFO - Sequenced data shape: (1935655, 12, 7), Target shape: (1935655, 4)
2025-07-02 15:15:04,251 - INFO - Train shape: (1548524, 12, 7), Validation shape: (387131, 12, 7)
2025-07-02 15:15:04,252 - INFO - Building LSTM model...


2025-07-02 15:15:04,307 - INFO - Model: "sequential_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 20,644 (80.64 KB)
 Trainable params: 20,644 (80.64 KB)
 Non-trainable params: 0 (0.00 B)

2025-07-02 15:15:04,307 - INFO - Training model...


Epoch 1/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 69s 22ms/step - auc: 0.7098 - loss: 0.0378 - val_auc: 0.7908 - val_loss: 0.0114
Epoch 2/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 68s 22ms/step - auc: 0.7894 - loss: 0.0113 - val_auc: 0.7595 - val_loss: 0.0114
Epoch 3/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 66s 22ms/step - auc: 0.8079 - loss: 0.0112 - val_auc: 0.8021 - val_loss: 0.0109
Epoch 4/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 67s 22ms/step - auc: 0.8155 - loss: 0.0108 - val_auc: 0.8212 - val_loss: 0.0108
Epoch 5/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 68s 22ms/step - auc: 0.8214 - loss: 0.0107 - val_auc: 0.8218 - val_loss: 0.0108
Epoch 6/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 68s 22ms/step - auc: 0.8227 - loss: 0.0104 - val_auc: 0.8378 - val_loss: 0.0106
Epoch 7/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 68s 22ms/step - auc: 0.8320 - loss: 0.0103 - val_auc: 0.8303 - val_loss: 0.0105
Epoch 8/15
3025/3025 ━━━━━━━━━━━━━━━━━━━━ 67s 22ms/step - auc: 0.8342 - loss: 0.0103 - val_auc: 0.8280 - val_loss: 0.0104
Epoch 9/15
3025/3025 ━━━

2025-07-02 15:25:32,319 - INFO - 
Validation Performance:
2025-07-02 15:25:32,784 - INFO - oestrus (AUC=0.885, Threshold=0.139):
2025-07-02 15:25:32,785 - INFO -   Precision: 0.214, Recall: 0.241, F1: 0.227
2025-07-02 15:25:33,253 - INFO - calving (AUC=0.973, Threshold=0.253):
2025-07-02 15:25:33,253 - INFO -   Precision: 0.339, Recall: 0.382, F1: 0.359
2025-07-02 15:25:33,717 - INFO - lameness (AUC=0.852, Threshold=0.077):
2025-07-02 15:25:33,718 - INFO -   Precision: 0.152, Recall: 0.172, F1: 0.161
2025-07-02 15:25:34,191 - INFO - mastitis (AUC=0.785, Threshold=0.038):
2025-07-02 15:25:34,191 - INFO -   Precision: 0.055, Recall: 0.019, F1: 0.028
2025-07-02 15:25:34,220 - INFO - Model saved successfully!
2025-07-02 15:25:34,379 - INFO - 
Final Model Performance:
2025-07-02 15:25:34,380 - INFO - oestrus: AUC=0.885, F1=0.227, Precision=0.214, Recall=0.241
2025-07-02 15:25:34,380 - INFO - calving: AUC=0.973, F1=0.359, Precision=0.339, Recall=0.382
2025-07-02 15:25:34,381 - INFO - lamenes